In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

def american_option_discrete_div(S, K, T, r, sigma, option_type, div_dates, div_amts, steps):
    """
    Calculate American option price with discrete dividends using binomial tree
    """
    dt = T / steps
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)
    discount = np.exp(-r * dt)

    # Convert dividend dates to step indices
    div_steps = [int(div_date * steps) for div_date in div_dates]

    # Build the tree
    asset_tree = np.zeros((steps + 1, steps + 1))
    asset_tree[0, 0] = S

    for i in range(1, steps + 1):
        # Check if dividend is paid at this step
        div_paid = sum(div_amts[j] for j, ds in enumerate(div_steps) if ds == i)

        for j in range(i + 1):
            if j == 0:
                asset_tree[i, j] = asset_tree[i-1, j] * u - div_paid
            elif j == i:
                asset_tree[i, j] = asset_tree[i-1, j-1] * d - div_paid
            else:
                asset_tree[i, j] = asset_tree[i-1, j-1] * d - div_paid

    # Initialize option values at maturity
    option_values = np.zeros(steps + 1)
    for i in range(steps + 1):
        ST = S
        for j in range(steps):
            if j < i:
                ST *= u
            else:
                ST *= d
            # Subtract dividends paid before maturity
            for k, ds in enumerate(div_steps):
                if ds <= j + 1:
                    ST -= div_amts[k]

        if option_type == 'Call':
            option_values[i] = max(ST - K, 0)
        else:
            option_values[i] = max(K - ST, 0)

    # Recalculate with proper tree structure
    ST_values = np.zeros(steps + 1)
    for i in range(steps + 1):
        ST_values[i] = S * (u ** (steps - i)) * (d ** i)
        for k, ds in enumerate(div_steps):
            if ds <= steps:
                ST_values[i] -= div_amts[k] * (u ** max(0, steps - ds - i)) * (d ** min(i, steps - ds))

    if option_type == 'Call':
        option_values = np.maximum(ST_values - K, 0)
    else:
        option_values = np.maximum(K - ST_values, 0)

    # Backward induction
    for j in range(steps - 1, -1, -1):
        for i in range(j + 1):
            S_node = S * (u ** (j - i)) * (d ** i)

            # Subtract dividends already paid
            for k, ds in enumerate(div_steps):
                if ds <= j:
                    S_node -= div_amts[k]

            option_values[i] = discount * (p * option_values[i] + (1 - p) * option_values[i + 1])

            # Early exercise
            if option_type == 'Call':
                exercise_value = max(S_node - K, 0)
            else:
                exercise_value = max(K - S_node, 0)

            option_values[i] = max(option_values[i], exercise_value)

    return option_values[0]

def american_option_greeks_discrete(S, K, T, r, sigma, option_type, div_dates, div_amts):
    """
    Calculate American option price and Greeks with discrete dividends
    """
    steps = max(100, int(T * 365))

    price = american_option_discrete_div(S, K, T, r, sigma, option_type, div_dates, div_amts, steps)

    dS = S * 0.01
    price_up = american_option_discrete_div(S + dS, K, T, r, sigma, option_type, div_dates, div_amts, steps)
    price_down = american_option_discrete_div(S - dS, K, T, r, sigma, option_type, div_dates, div_amts, steps)
    delta = (price_up - price_down) / (2 * dS)
    gamma = (price_up - 2 * price + price_down) / (dS ** 2)

    dsigma = 0.01
    price_sigma_up = american_option_discrete_div(S, K, T, r, sigma + dsigma, option_type, div_dates, div_amts, steps)
    vega = (price_sigma_up - price) / dsigma

    dr = 0.01
    price_r_up = american_option_discrete_div(S, K, T, r + dr, sigma, option_type, div_dates, div_amts, steps)
    rho = (price_r_up - price) / dr

    dT = 1/365
    if T > dT:
        # Adjust dividend dates for time shift
        div_dates_shifted = [max(0, d - dT) for d in div_dates]
        price_T_down = american_option_discrete_div(S, K, T - dT, r, sigma, option_type, div_dates_shifted, div_amts, steps)
        theta = (price_T_down - price) / dT
    else:
        theta = 0

    return price, delta, gamma, vega, rho, theta

# Read data
DATA_DIR = Path.cwd() / "testfiles_" / "data"
CSV_PATH = DATA_DIR / "test12_3.csv"
df = pd.read_csv(CSV_PATH, header=0)

df = df.dropna(subset=['ID'])

results = []
for _, row in df.iterrows():
    S = row['Underlying']
    K = row['Strike']
    T = row['DaysToMaturity'] / row['DayPerYear']
    r = row['RiskFreeRate']
    sigma = row['ImpliedVol']
    option_type = row['Option Type']

    # Parse dividend dates and amounts
    div_dates_str = str(row['DividendDates']).split(',')
    div_amts_str = str(row['DividendAmts']).split(',')

    div_dates = [float(d.strip()) / row['DayPerYear'] for d in div_dates_str]
    div_amts = [float(a.strip()) for a in div_amts_str]

    value, delta, gamma, vega, rho, theta = american_option_greeks_discrete(
        S, K, T, r, sigma, option_type, div_dates, div_amts
    )

    results.append({
        'ID': int(row['ID']),
        'Value': value
    })

output_df = pd.DataFrame(results)
print(output_df)

   ID      Value
0   1  14.504598
1   2  11.777884
